# 5. 5. Asignación de carteros

Una Oficina de Correos necesita un número distinto de carteros y carteras a tiempo completo según el dı́a de la semana. 
La cantidad requerida cada dı́a, calculada a partir del número de envı́os por entregar, se detalla en la Tabla 3.

`b = [17, 13, 15, 19, 14, 16, 11]`

Según las reglas del sindicato, cada empleado a tiempo completo debe trabajar cinco dı́as seguidos y descansar los dos siguientes. 
Por ejemplo, quien trabaje de lunes a viernes descansará el sábado y el domingo. 
La Oficina de Correos desea cubrir sus requerimientos diarios empleando únicamente trabajadores a tiempo completo.

**a)** Formule un modelo de programación entera que permita a la Oficina de Correos determinar el número mı́nimo de carteros y carteras a tiempo completo con el que debe operar.


In [1]:
from docplex.mp.model import Model
import numpy as np

mdl = Model(name='Asignacion de carteros')

## Datos y supuestos  
* numero distinto de carteros por semana
* dicho numero depende de la cantidad de envios por entregar ese día (dado en tabla)
* se debe trabajar 5 dias seguidos y dos de descanso
* numero minimo de carteros

## Formulación matemática 
### Variables de decisión

$$x_{i} = \text{\# de trbajadores que usan el patron i}$$
definimos patrones 
cuantos patrones? -> # de días de inicio posibles
$$i \epsilon \{0, 1, 2, 3, 4, 5, 6\} = \{\text{LMMiJV}, \text{MMiJVS} \dots \}$$
#### Matriz de cobertura 
* El patron $i$ cubre el día $j$ ? 
$$ j \epsilon \{0, 1, 2, 3, 4, 5, 6\} = \{L, M, Mi, J, V, S, D\}$$
$$A = [a_{ij}] = \
\begin{pmatrix}
1 & 1 & 1 & 1 & 1 &0 &0 \\ 
0 & 1 & 1 & 1 & 1 &1 &0 \\ 
0 & 0 & 1 & 1 & 1 &1 &1 \\ 
1 & 0 & 0 & 1 & 1 &1 &1 \\
1 & 1 & 0 & 0 & 1 &1 &1 \\ 
1 & 1 & 1 & 0 & 0 &1 &1 \\
1 & 1 & 1 & 1 & 0 &0 &1
\end{pmatrix}$$



In [2]:
dias = [0, 1, 2, 3, 4, 5, 6]

# Variable de decision
x = mdl.integer_var_list(len(dias), name='x')  # por defecto, x_i >= 0 y sin límite superior

a = [
    [1, 1, 1, 1, 1, 0, 0],
    [0, 1, 1, 1, 1, 1, 0],
    [0, 0, 1, 1, 1, 1, 1],
    [1, 0, 0, 1, 1, 1, 1],
    [1, 1, 0, 0, 1, 1, 1], 
    [1, 1, 1, 0, 0, 1, 1], 
    [1, 1, 1, 1, 0, 0, 1]
]

a_np = np.array(a)

In [3]:
x

[docplex.mp.Var(type=I,name='x_0'),
 docplex.mp.Var(type=I,name='x_1'),
 docplex.mp.Var(type=I,name='x_2'),
 docplex.mp.Var(type=I,name='x_3'),
 docplex.mp.Var(type=I,name='x_4'),
 docplex.mp.Var(type=I,name='x_5'),
 docplex.mp.Var(type=I,name='x_6')]

### Restricciones

#### Restricciones 1: demanda de personal
* dado un día fijo $j$, la suma de personal debe satisfacer la demanda de dicho día
* personal por día = (trabajadores que usan un patron) * (# de patrones que cubren el dia)

$$ \sum_i x_i a_{ij} == b_j \ \ \forall \ j$$
donde las demandas son 
$$b_j = [17, 13, 15, 19, 14, 16, 11]$$
#### Restricciones 2: 
* se debe trabajar 5 dias seguidos y dos de descanso
* la suma de los días que se trabajan debe ser $=5$ para cada trabajador 
$$\text{ya está impuesta desde el planteamiento de la variable de decisión  $x_i$ }$$


In [4]:
# Restricciones: demanda
b = [17, 13, 15, 19, 14, 16, 11]

mdl.add_constraints(
	mdl.sum(x[i]*a_np[i][j] for i in dias) >= b[j] for j in range(len(b))
)

mdl.minimize(
	mdl.sum(x[i] for i in dias)
	)

solution = mdl.solve(log_output=True,)

print(solution)

Version identifier: 22.2.0.0 | 2026-05-04 | 5dff98e58
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 46.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
Reduced MIP has 7 rows, 7 columns, and 35 nonzeros.
Reduced MIP has 0 binaries, 7 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.02 sec. (0.01 ticks)
Tried aggregator 1 time.
Detecting symmetries...
Reduced MIP has 7 rows, 7 columns, and 35 nonzeros.
Reduced MIP has 0 binaries, 7 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.01 sec. (0.02 ticks)
MIP emphasis: balance optimality and feasibility.
MIP search method: dynamic search.
Parallel mode: deterministic, using up to 4 threads.
Root relaxation solution time = 0.01 sec. (0.01 ticks)

        Nodes                                         Cuts/
   Node  Left     Objective  IInf  Best Integer    Best Bound    ItCnt     Gap

*     0+    0                           46.0000        0.0000           100.00%
*     0+    0                

Total (root+branch&cut) =    0.11 sec. (0.07 ticks)
solution for: Asignacion de carteros
objective: 23
status: OPTIMAL_SOLUTION(2)
x_0=6
x_1=4
x_2=1
x_3=8
x_5=4

